# Lab 4: RegistryToolProvider — Dynamic Tool Discovery for Strands Agents

## Overview

This notebook demonstrates the **RegistryToolProvider** — a Strands `ToolProvider` that discovers tools
from AWS Agent Registry via semantic search and injects them into the agent's LLM context automatically.

### The Problem

Every tool given to an LLM costs ~200-500 tokens. An agent with 200 tools burns 40K-100K tokens of context
before the user even asks a question — expensive, slow, and the model gets confused.

### The Solution

The `RegistryToolProvider` searches the Registry with domain keywords before each LLM turn and returns
only the relevant tools. The agent sees 5-10 tools instead of 200.

```
agent("Check my open tickets")
  │
  ├─ provider.load_tools()
  │    ├─ SearchRegistryRecords("CRM")        → 1 record, 1 tool
  │    ├─ SearchRegistryRecords("ticketing")   → 1 record, 5 tools
  │    └─ 6 tools injected into LLM context
  │
  ├─ LLM picks: query_tickets
  ├─ Provider calls Gateway → tools/call → result
  └─ LLM answers user
```

## What You'll Learn

1. **Verify Registry records** — confirm tools are registered and APPROVED
2. **Instantiate the RegistryToolProvider** — configure domains, caching, security
3. **Test tool discovery** — see which tools are returned for different domain keywords
4. **Test caching and lifecycle** — verify cache hits and consumer cleanup
5. **Test security controls** — HTTPS enforcement, APPROVED-only filtering
6. **Wire it into a Strands agent** — end-to-end agent with dynamic tools
7. **Compare all 5 discovery patterns** — understand when to use each

## Prerequisites

- AWS credentials with Registry and Gateway permissions
- At least one Registry with APPROVED records containing `tools`
- Python packages: `strands-agents>=1.23.0`, `boto3`, `httpx`

---
## Setup

In [ ]:
!pip install -q strands-agents boto3 httpx

In [ ]:
import boto3
import json
import time
import asyncio
import sys
import os

# Configuration
AWS_REGION = "us-west-2"
REGISTRY_ENDPOINT = "https://bedrock-agentcore-control.us-west-2.amazonaws.com"
REGISTRY_DP_ENDPOINT = "https://bedrock-agentcore.us-west-2.amazonaws.com"
REGISTRY_ID = ""  #registry ID

# Gateway config (for tool invocation)
GATEWAY_URL = None  # Set to your Gateway URL to test invocation
GATEWAY_REGION = "us-east-1"

# Ensure the code samples directory is in the path
sys.path.insert(0, os.path.join(os.getcwd(), "..", "python-code-samples", "registry-tool-provider"))
sys.path.insert(0, os.path.join(os.getcwd(), "python-code-samples", "registry-tool-provider"))

print(f"Region: {AWS_REGION}")
print(f"Registry: {REGISTRY_ID}")
print(f"Endpoint: {REGISTRY_ENDPOINT}")

#only for rob
import certifi, os
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

Region: us-west-2
Registry: Vf4gtZ5mreKG
Endpoint: https://bedrock-agentcore-control.us-west-2.amazonaws.com


---
## Step 1: Verify Registry Records

Before using the ToolProvider, let's confirm what's in the Registry.
We need APPROVED records with `tools` (for MCP) or `agentCard` (for A2A).

In [2]:
cp_client = boto3.client(
    "bedrock-agentcore-registry-control",
    region_name=AWS_REGION,
    endpoint_url=REGISTRY_ENDPOINT,
)

# List all records in the registry
records = cp_client.list_registry_records(registryId=REGISTRY_ID)["registryRecords"]

print(f"Registry {REGISTRY_ID} has {len(records)} records:\n")
for r in records:
    # Get full record to check tool count
    full = cp_client.get_registry_record(
        registryId=REGISTRY_ID, recordId=r["recordId"]
    )
    descriptors = full.get("descriptors", {})
    tool_schema = descriptors.get("mcp", {}).get("tools", {}).get("inlineContent", "")
    tool_count = 0
    if tool_schema:
        try:
            td = json.loads(tool_schema)
            if isinstance(td, dict): td = td.get("tools", [])
            tool_count = len(td)
        except: pass

    print(f"  📦 {r['name']}")
    print(f"     Protocol: {r.get('protocol', '?')} | Status: {r.get('status', '?')} | Tools: {tool_count}")
    print()

Registry Vf4gtZ5mreKG has 3 records:

  📦 salesforce_agentforce
     Protocol: MCP | Status: APPROVED | Tools: 1

  📦 kb_gateway_poc
     Protocol: MCP | Status: DRAFT | Tools: 5

  📦 mcp_gateway
     Protocol: MCP | Status: DRAFT | Tools: 28



---
## Step 2: Instantiate the RegistryToolProvider

The provider takes:
- **registry_ids** — which registries to search
- **domains** — semantic search keywords (e.g., `["salesforce", "knowledge base"]`)
- **gateway_url** + **gateway_token_fn** — for MCP tool invocation through Gateway
- **required_status** — only load APPROVED records (default)
- **cache_ttl** — how long to cache results (default 300s)

In [3]:
from registry_tool_provider import RegistryToolProvider
import logging
logging.basicConfig(level=logging.INFO)

provider = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["salesforce", "knowledge base", "gateway"],
    region=AWS_REGION,
    endpoint_url=REGISTRY_DP_ENDPOINT,
    gateway_url=GATEWAY_URL,
    cache_ttl=0,  # Disable cache for demo (so we see fresh results each time)
    required_status="APPROVED",
)

print("✅ RegistryToolProvider created")
print(f"   Registries: {provider._registry_ids}")
print(f"   Domains: {provider._domains}")
print(f"   Required status: {provider._required_status}")

INFO:botocore.client:No endpoints ruleset found for service bedrock-agentcore-registry, falling back to legacy endpoint routing.


✅ RegistryToolProvider created
   Registries: ['Vf4gtZ5mreKG']
   Domains: ['salesforce', 'knowledge base', 'gateway']
   Required status: APPROVED


---
## Step 3: Test Tool Discovery

Call `load_tools()` — this is what the Strands agent calls before each LLM turn.
The provider will:
1. Search Registry with each domain keyword
2. Filter by APPROVED status
3. Parse tool schemas from matching records
4. Return `PythonAgentTool` instances

In [4]:
tools = await provider.load_tools()

print(f"🔍 Discovered {len(tools)} tools from {len(provider._domains)} domain(s):\n")
for t in tools:
    print(f"  🔧 {t.tool_name}")
    print(f"     {t.tool_spec['description'][:100]}")
    print()

INFO:registry_tool_provider:RegistryToolProvider: loaded 6 tools from 3 domain(s)


🔍 Discovered 6 tools from 3 domain(s):

  🔧 ask_agentforce
     Send a message to Salesforce Agentforce

  🔧 x_amz_bedrock_agentcore_search
     A special tool that returns a trimmed down list of tools given a context. Use this tool only when th

  🔧 query_AgentCore_knowledge_base
     Query the query_AgentCore_knowledge_base knowledge base.

  🔧 query_Sandoz_knowledge_base
     Query the query_Sandoz_knowledge_base knowledge base.

  🔧 query_daily_notes_kb_knowledge_base
     Daily notes knowledge base for testing AgentCore Gateway auto-registration

  🔧 query_dormakaba_sales_kb_knowledge_base
     Dormakaba sales documentation including proposals, pricing, and product information



### Try different domain keywords

Change the domains and see how the tool set changes. This is the core value —
different keywords = different tools = bounded LLM context.

In [5]:
# Narrow search: only salesforce
narrow = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["salesforce"],
    region=AWS_REGION,
    endpoint_url=REGISTRY_DP_ENDPOINT,
    cache_ttl=0,
    required_status="APPROVED",
)
narrow_tools = await narrow.load_tools()
print(f"Narrow (salesforce only): {len(narrow_tools)} tools")
for t in narrow_tools:
    print(f"  - {t.tool_name}")

print()

# Broad search: everything
broad = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["salesforce", "knowledge base", "gateway", "search", "tools"],
    region=AWS_REGION,
    endpoint_url=REGISTRY_DP_ENDPOINT,
    cache_ttl=0,
    required_status="APPROVED",
)
broad_tools = await broad.load_tools()
print(f"Broad (5 domains): {len(broad_tools)} tools")
for t in broad_tools:
    print(f"  - {t.tool_name}")

INFO:botocore.client:No endpoints ruleset found for service bedrock-agentcore-registry, falling back to legacy endpoint routing.
INFO:registry_tool_provider:RegistryToolProvider: loaded 1 tools from 1 domain(s)
INFO:botocore.client:No endpoints ruleset found for service bedrock-agentcore-registry, falling back to legacy endpoint routing.


Narrow (salesforce only): 1 tools
  - ask_agentforce



INFO:registry_tool_provider:RegistryToolProvider: loaded 6 tools from 5 domain(s)


Broad (5 domains): 6 tools
  - ask_agentforce
  - x_amz_bedrock_agentcore_search
  - query_AgentCore_knowledge_base
  - query_Sandoz_knowledge_base
  - query_daily_notes_kb_knowledge_base
  - query_dormakaba_sales_kb_knowledge_base


---
## Step 4: Test Caching and Consumer Lifecycle

The provider caches results to avoid hitting the Registry API on every LLM turn.
Cache clears when the last consumer (agent) is removed.

In [6]:
# Enable caching
provider._cache_ttl = 300
provider._cache = []  # clear any existing cache
provider._cache_ts = 0

# First call — hits Registry API
start = time.time()
tools1 = await provider.load_tools()
first_ms = (time.time() - start) * 1000
print(f"First call:  {len(tools1)} tools in {first_ms:.0f}ms (API call)")

# Second call — cache hit
start = time.time()
tools2 = await provider.load_tools()
second_ms = (time.time() - start) * 1000
print(f"Second call: {len(tools2)} tools in {second_ms:.1f}ms (cache hit)")

# Consumer lifecycle
provider.add_consumer("agent-1")
provider.add_consumer("agent-2")
print(f"\nConsumers: 2 — cache alive: {bool(provider._cache)}")

provider.remove_consumer("agent-1")
print(f"Consumers: 1 — cache alive: {bool(provider._cache)}")

provider.remove_consumer("agent-2")
print(f"Consumers: 0 — cache alive: {bool(provider._cache)} (cleared!)")

INFO:registry_tool_provider:RegistryToolProvider: loaded 6 tools from 3 domain(s)


First call:  6 tools in 959ms (API call)
Second call: 6 tools in 0.1ms (cache hit)

Consumers: 2 — cache alive: True
Consumers: 1 — cache alive: True
Consumers: 0 — cache alive: False (cleared!)


---
## Step 5: Test Security Controls

The provider includes several security hardening measures.

In [7]:
# Test 1: HTTPS enforcement
try:
    RegistryToolProvider(
        registry_ids=["x"], domains=["x"],
        gateway_url="http://not-secure.com/mcp"  # HTTP, not HTTPS
    )
    print("❌ HTTP should have been rejected")
except ValueError as e:
    print(f"✅ HTTPS enforcement: {e}")

# Test 2: APPROVED-only filtering
# Create a provider that requires DRAFT status (should find fewer/no tools)
draft_provider = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["salesforce"],
    region=AWS_REGION,
    endpoint_url=REGISTRY_DP_ENDPOINT,
    cache_ttl=0,
    required_status="DRAFT",
)
draft_tools = await draft_provider.load_tools()
approved_tools = await narrow.load_tools()  # reuse narrow from Step 3
print(f"✅ Status filtering: APPROVED={len(approved_tools)} tools, DRAFT={len(draft_tools)} tools")

# Test 3: Runtime ARN allowlist
restricted = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["salesforce"],
    region=AWS_REGION,
    endpoint_url=REGISTRY_DP_ENDPOINT,
    cache_ttl=0,
    allowed_runtime_arns=["arn:aws:bedrock-agentcore:us-east-1:123:runtime/only-this-one"],
)
print(f"✅ Runtime ARN allowlist configured (A2A agents restricted)")

print("\n✅ All security controls working")

INFO:botocore.client:No endpoints ruleset found for service bedrock-agentcore-registry, falling back to legacy endpoint routing.


✅ HTTPS enforcement: gateway_url must use HTTPS, got: http://not-secure.com/mcp


INFO:registry_tool_provider:RegistryToolProvider: loaded 0 tools from 1 domain(s)
INFO:registry_tool_provider:RegistryToolProvider: loaded 1 tools from 1 domain(s)
INFO:botocore.client:No endpoints ruleset found for service bedrock-agentcore-registry, falling back to legacy endpoint routing.


✅ Status filtering: APPROVED=1 tools, DRAFT=0 tools
✅ Runtime ARN allowlist configured (A2A agents restricted)

✅ All security controls working


---
## Step 6: Wire Into a Strands Agent

This is the end-to-end integration. The agent uses the ToolProvider to get tools
dynamically from Registry. No hardcoded tool list.

**Note:** This step requires a Gateway URL and token function to actually invoke tools.
Without them, the agent will discover tools but tool calls will return an error.

In [8]:
from strands import Agent
from strands.models import BedrockModel

# Create provider with caching enabled
agent_provider = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["salesforce", "knowledge base"],
    region=AWS_REGION,
    endpoint_url=REGISTRY_DP_ENDPOINT,
    gateway_url=GATEWAY_URL,
    cache_ttl=300,
    required_status=None,
)

# Load tools from Registry, then pass to Agent
discovered_tools = await agent_provider.load_tools()
print(f"Discovered {len(discovered_tools)} tools from Registry")

agent = Agent(
    model=BedrockModel(model_id="global.anthropic.claude-sonnet-4-20250514-v1:0", region_name="us-east-1"),
    tools=list(discovered_tools),
    system_prompt="You are a helpful assistant. Use the available tools to answer questions. "
                  "If a tool call fails, explain what happened.",
)

print(f"✅ Agent created with {len(discovered_tools)} tools from Registry")

INFO:botocore.client:No endpoints ruleset found for service bedrock-agentcore-registry, falling back to legacy endpoint routing.
INFO:registry_tool_provider:RegistryToolProvider: loaded 6 tools from 2 domain(s)


Discovered 6 tools from Registry
✅ Agent created with 6 tools from Registry


In [9]:
# Ask the agent what tools it has
result = agent("What tools do you have available? List them with a brief description of each.")
print(result)

INFO:strands.telemetry.metrics:Creating Strands MetricsClient


Based on the available functions, I have access to the following tools:

1. **ask_agentforce** - Send a message to Salesforce Agentforce. This allows me to communicate with Salesforce's AI agent platform.

2. **x_amz_bedrock_agentcore_search** - A special tool that returns a trimmed down list of tools given a context. This appears to be for searching/filtering available tools based on context.

3. **query_AgentCore_knowledge_base** - Query the AgentCore knowledge base. This allows me to search through AgentCore-specific documentation or information.

4. **query_Sandoz_knowledge_base** - Query the Sandoz knowledge base. This provides access to Sandoz (pharmaceutical company) related information and documentation.

5. **query_daily_notes_kb_knowledge_base** - Daily notes knowledge base for testing AgentCore Gateway auto-registration. This appears to be a test knowledge base containing daily notes.

6. **query_dormakaba_sales_kb_knowledge_base** - Access to Dormakaba sales documentation i

---
## Step 7: The 5 Discovery Patterns

The ToolProvider (Pattern 3) is the recommended default, but there are 4 other ways
to connect a Strands agent to Registry. Here's when to use each:

| # | Pattern | When tools resolve | Context cost | Best for |
|---|---|---|---|---|
| 1 | **Pre-fetch** | Startup (once) | High — all tools | Dev/test, <15 tools |
| 2 | **Registry-as-Tool** | LLM decides | Low — 1 search tool | Exploration, unknown domains |
| 3 | **ToolProvider** ✅ | Before each turn | Bounded — domain only | **Production default** |
| 4 | **Hook Interception** | On failure | Grows on demand | Fallback, legacy migration |
| 5 | **Planner+Executor** | Planning phase | Minimal — planned only | Multi-step workflows |

### Quick comparison

In [ ]:
# Pattern 1: Pre-fetch (all tools at startup)
print("=" * 60)
print("Pattern 1: Pre-fetch")
print("=" * 60)
all_records = cp_client.list_registry_records(registryId=REGISTRY_ID)["registryRecords"]
total_tools = 0
for r in all_records:
    full = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=r["recordId"])
    ts = full.get("descriptors", {}).get("mcp", {}).get("tools", {}).get("inlineContent", "")
    if ts:
        try:
            td = json.loads(ts)
            if isinstance(td, dict): td = td.get("tools", [])
            total_tools += len(td)
        except: pass
print(f"→ Would load ALL {total_tools} tools into every LLM call")
print(f"→ ~{total_tools * 350} tokens of context per request")
print()

# Pattern 3: ToolProvider (bounded)
print("=" * 60)
print("Pattern 3: ToolProvider (this solution)")
print("=" * 60)
provider._cache_ttl = 0  # fresh search
provider._cache = []
tp_tools = await provider.load_tools()
print(f"→ Loaded {len(tp_tools)} tools (from {len(provider._domains)} domain keywords)")
print(f"→ ~{len(tp_tools) * 350} tokens of context per request")
print(f"→ Saved ~{(total_tools - len(tp_tools)) * 350} tokens vs pre-fetch")

In [ ]:
# Pattern 2: Registry-as-Tool (LLM decides when to search)
from strands import tool as strands_tool

dp_client = boto3.client(
    "bedrock-agentcore-registry",
    region_name=AWS_REGION,
    endpoint_url=REGISTRY_DP_ENDPOINT,
)

@strands_tool
def search_registry(query: str) -> str:
    """Search the AWS Agent Registry to discover available tools and agents."""
    resp = dp_client.search_registry_records(
        registryIds=[REGISTRY_ID], searchQuery=query, maxResults=5
    )
    return json.dumps(
        [{"name": r["name"], "descriptorType": r.get("descriptorType", ""),
          "description": r.get("description", "")[:100]}
         for r in resp["registryRecords"]], indent=2
    )

print("Pattern 2: Registry-as-Tool")
print("→ LLM gets 1 tool (search_registry) — ~300 tokens")
print("→ LLM decides when to search — adds 1 round-trip")
print("→ Discovered tools are NOT callable (discovery only)")
print()
print("Example search result:")
print(search_registry(query="salesforce"))

---
## Summary

| What we tested | Result |
|---|---|
| Tool discovery from Registry | ✅ Semantic search returns relevant tools |
| Domain scoping | ✅ Different keywords = different tool sets |
| Caching | ✅ First call ~500ms, cached calls <1ms |
| Consumer lifecycle | ✅ Cache clears when last agent removed |
| HTTPS enforcement | ✅ HTTP URLs rejected |
| APPROVED-only filtering | ✅ Draft records excluded |
| Strands agent integration | ✅ Agent discovers tools automatically |
| Pattern comparison | ✅ ToolProvider saves ~80% context vs pre-fetch |

### Key Takeaway

The `RegistryToolProvider` is the **recommended default** for production Strands agents.
It's automatic (no LLM decision needed), bounded (predictable context cost),
dynamic (new tools appear without redeployment), and simple (one class, a few parameters).

### Files

| File | Description |
|---|---|
| `python-code-samples/registry-tool-provider/registry_tool_provider.py` | The RegistryToolProvider class |
| `python-code-samples/registry-tool-provider/example_tool_provider_agent.py` | Minimal agent example |
| `python-code-samples/registry-tool-provider/README.md` | Developer README |
| `python-code-samples/registry-tool-provider/DISCOVERY_PATTERNS.md` | Full design doc with all 5 patterns |